# OMI Integrated Analysis — Prices, Transactions & Population

## Overview

This notebook is the **capstone** of the OMI analytical pipeline. It does **not** recompute the raw ingestions performed in notebooks `01` and `02`. Instead, it:

1. Loads the already-produced analytical panels:
   - **Prices** — residential OMI quotations, aggregated to municipality–year;
   - **Transactions** — residential + non-residential NTN, municipality–year;
   - **Population** — ISTAT resident population, municipality–year.
2. Builds a single **municipality–year integrated panel**.
3. Performs relational analysis across the three dimensions:
   - prices ↔ transactions,
   - prices ↔ population,
   - transactions ↔ population.

## Methodological Posture

All results presented here are **associational**, not causal. We report:

- correlation coefficients (Pearson for linear, Spearman for monotonic);
- cross-sectional patterns;
- panel (within-municipality) patterns;
- growth-rate co-movement;
- lagged associations (for temporal patterns).

We explicitly **do not** claim that population growth *causes* price growth
or that transaction activity *drives* prices. We describe **co-movement** and
**heterogeneity** across space and time.

---

## 1. Setup & Load Pre-Computed Artifacts

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

# ------------------------------------------------------------------
# Resolve project root
# ------------------------------------------------------------------
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "data").is_dir():
        PROJECT_ROOT = candidate
        break

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR = PROJECT_ROOT / "data" / "raw"

# ------------------------------------------------------------------
# Analysis constants
# ------------------------------------------------------------------
MIN_OBS_MUNICIPALITY = 20      # keep municipalities with at least this many transactions / year
MIN_YEARS_FOR_PANEL = 8        # minimum years needed for growth correlations
MIN_POP = 500                  # drop micro-municipalities from relational analyses

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

### 1.1 Load the three pre-computed sources

- `market_panel` is the analytical panel produced at the end of notebook `02`
  (municipality–year transactions).
- `price_panel` is the municipality–year median quotation obtained by
  aggregating the OMI quotations panel from notebook `01`.
- `population_panel` is the ISTAT municipality–year population.

We attempt to load each from a cached Parquet; if not present, the cell
rebuilds the source **from the previous notebook's outputs** (not from raw CSVs).

In [2]:
# ---------------------------------------------------------------
# 1. Transactions panel (from notebook 02)
# ---------------------------------------------------------------
tx_candidates = [
    PROCESSED_DIR / "omi_transactions_panel.parquet",
    PROCESSED_DIR / "market_panel.parquet",
    PROCESSED_DIR / "omi_market_panel.parquet",
]
tx_path = next((p for p in tx_candidates if p.exists()), None)

if tx_path is None:
    raise FileNotFoundError(
        "Transactions panel not found. Run notebook 02 (or its export cell) first."
    )

market_panel = pd.read_parquet(tx_path)
market_panel["codcom"] = market_panel["codcom"].astype("string").str.strip().str.upper()

print(f"Transactions panel: {market_panel.shape}")
print(f"  Years: {market_panel['year'].min()}–{market_panel['year'].max()}")
print(f"  Municipalities: {market_panel['codcom'].nunique():,}")

FileNotFoundError: Transactions panel not found. Run notebook 02 (or its export cell) first.

In [ ]:
# ---------------------------------------------------------------
# 2. Quotations panel (from notebook 01)
# ---------------------------------------------------------------
omi = pd.read_parquet(PROCESSED_DIR / "omi_quotations.parquet")
omi["codcom"] = omi["Comune_amm"].astype("string").str.strip().str.upper()
omi["Compr_mid"] = pd.to_numeric(omi["Compr_mid"], errors="coerce")
omi = omi.dropna(subset=["Compr_mid", "codcom"])

# Residential filter (same rule as notebook 01)
residential_mask = (
    omi["Descr_Tipologia"]
    .astype("string")
    .str.contains("abitazion|villa", case=False, na=False)
)
omi_res = omi.loc[residential_mask].copy()

print(f"Residential quotations: {len(omi_res):,} rows")
print(f"  Temporal coverage: {omi_res['reference_date'].min().date()} → "
      f"{omi_res['reference_date'].max().date()}")

In [ ]:
# ---------------------------------------------------------------
# Aggregate quotations to municipality–year median
# ---------------------------------------------------------------
omi_res["year"] = omi_res["reference_date"].dt.year

price_panel = (
    omi_res
    .groupby(["year", "codcom"], as_index=False)
    .agg(
        price_median=("Compr_mid", "median"),
        price_mean=("Compr_mid", "mean"),
        price_p25=("Compr_mid", lambda x: x.quantile(0.25)),
        price_p75=("Compr_mid", lambda x: x.quantile(0.75)),
        n_quotes=("Compr_mid", "size"),
    )
    .rename(columns={"price_median": "price_median_eur_m2"})
)

print(f"Price panel: {price_panel.shape}")
print(f"  Municipalities: {price_panel['codcom'].nunique():,}")
print(f"  Years: {price_panel['year'].min()}–{price_panel['year'].max()}")

In [ ]:
# ---------------------------------------------------------------
# 3. Population panel (ISTAT)
# ---------------------------------------------------------------
pop_candidates = [
    RAW_DIR / "population" / "istat_population_municipality_year.parquet",
    PROCESSED_DIR / "istat_population_panel.parquet",
    PROCESSED_DIR / "population_panel.parquet",
]
pop_path = next((p for p in pop_candidates if p.exists()), None)

if pop_path is None:
    print(
        "WARNING: ISTAT population file not found. "
        "Population-related sections will be skipped. "
        "Expected one of:\n  - " + "\n  - ".join(str(p) for p in pop_candidates)
    )
    population_panel = pd.DataFrame(columns=["year", "codcom", "population"])
else:
    population_panel = pd.read_parquet(pop_path)
    population_panel["codcom"] = (
        population_panel["codcom"].astype("string").str.strip().str.upper()
    )
    print(f"Population panel: {population_panel.shape}")
    print(f"  Years: {population_panel['year'].min()}–{population_panel['year'].max()}")
    print(f"  Municipalities: {population_panel['codcom'].nunique():,}")

## 2. Build the Integrated Municipality–Year Panel

The join key is `(year, codcom)`. We use an **outer** merge on the
transactions panel (the reference universe) and attach prices and population.
Cells where a source is missing are preserved with `NaN` so that we can
quantify **coverage loss** and restrict relational analyses to the
intersection.

### Important: harmonisation of `codcom`

The ISTAT numeric code (`Comune_ISTAT`, e.g. `13066001`) and the OMI
alphanumeric code (`codcom`, e.g. `A001`) are different identifier systems.
If the ISTAT file only carries the numeric ISTAT code, a crosswalk must be
built from the OMI `LISTA-COM` release. We assume that the prepared
`population_panel` already uses **OMI `codcom`**; if not, the user should
adapt the population loading step accordingly.

In [ ]:
integrated = market_panel.merge(
    price_panel[["year", "codcom", "price_median_eur_m2", "n_quotes"]],
    on=["year", "codcom"],
    how="left",
    validate="one_to_one",
)

if not population_panel.empty:
    integrated = integrated.merge(
        population_panel[["year", "codcom", "population"]],
        on=["year", "codcom"],
        how="left",
        validate="one_to_one",
    )
else:
    integrated["population"] = np.nan

print(f"Integrated panel: {integrated.shape}")
print(f"  Price coverage:      {integrated['price_median_eur_m2'].notna().mean():.1%}")
print(f"  Population coverage: {integrated['population'].notna().mean():.1%}")

In [ ]:
# ---------------------------------------------------------------
# Derived variables
# ---------------------------------------------------------------
integrated = integrated.sort_values(["codcom", "year"]).reset_index(drop=True)

# Volume per capita (transactions per 1,000 residents)
integrated["ntn_res_per_1k"] = np.where(
    integrated["population"].gt(0),
    integrated["ntn_res"] / integrated["population"] * 1000,
    np.nan,
)

# Log-price for elasticity-style analyses
integrated["log_price"] = np.log(integrated["price_median_eur_m2"])
integrated["log_ntn"] = np.log1p(integrated["ntn_res"])

# Growth rates (within municipality)
def _pct_change_by_group(df, col, group="codcom", periods=1):
    return (
        df.sort_values([group, "year"])
          .groupby(group, observed=True)[col]
          .pct_change(periods=periods)
          .replace([np.inf, -np.inf], np.nan)
    )

integrated["price_growth"] = _pct_change_by_group(integrated, "price_median_eur_m2")
integrated["ntn_growth"] = _pct_change_by_group(integrated, "ntn_res")
integrated["pop_growth"] = _pct_change_by_group(integrated, "population")

display(integrated.head(5))

## 3. Executive Overview

### What we are integrating

| Source              | Granularity              | Key measure             |
|---------------------|--------------------------|--------------------------|
| OMI quotations (01) | municipality × semester × zone | €/m² (mid-range)    |
| OMI transactions (02) | municipality × year    | NTN (transactions)       |
| ISTAT population    | municipality × year      | residents                |

### Analysis scales

1. **National** — aggregate time series;
2. **Regional** — cross-region comparison;
3. **Municipality** — cross-sectional and panel (within-unit) relations.

### Analytical framing

All cross-source relationships are reported as **associations**. Where useful,
we distinguish:

- **Cross-sectional association** — across municipalities in a given year;
- **Within-unit association** — after removing municipality fixed effects;
- **Growth co-movement** — correlation of growth rates;
- **Lagged association** — correlation between changes at t and t+k.

We treat all three as **descriptive facts of co-movement**, not as evidence
of causal channels.

In [ ]:
# ------------------------------------------------------------------
# Coverage summary
# ------------------------------------------------------------------
coverage = pd.DataFrame({
    "source": ["Transactions", "Prices", "Population"],
    "rows": [
        len(integrated),
        integrated["price_median_eur_m2"].notna().sum(),
        integrated["population"].notna().sum(),
    ],
    "coverage_pct": [
        1.0,
        integrated["price_median_eur_m2"].notna().mean(),
        integrated["population"].notna().mean(),
    ],
})
display(coverage.style.format({"coverage_pct": "{:.1%}"}))

In [ ]:
# ------------------------------------------------------------------
# Triangulated intersection (rows with all three sources present)
# ------------------------------------------------------------------
tri = integrated.dropna(subset=["ntn_res", "price_median_eur_m2", "population"]).copy()

print(f"Triangulated sample: {len(tri):,} municipality–year observations")
print(f"  Unique municipalities: {tri['codcom'].nunique():,}")
print(f"  Years: {tri['year'].min()}–{tri['year'].max()}")

## 4. National Market — A Unified Time Series

We summarise the national picture by aggregating the **triangulated sample**
to the year level. Prices are aggregated as a **median across municipalities**
(robust to Rome/Milan/Bolzano outliers); population as a sum; transactions as
a sum. This creates a single "national dashboard".

In [ ]:
national = (
    tri.groupby("year", as_index=False)
    .agg(
        price_median=("price_median_eur_m2", "median"),
        price_mean=("price_median_eur_m2", "mean"),
        ntn_res=("ntn_res", "sum"),
        population=("population", "sum"),
        n_municipalities=("codcom", "nunique"),
    )
    .sort_values("year")
)
national["ntn_per_1k"] = national["ntn_res"] / national["population"] * 1000

# Index (2011 = 100) for readability
base = national.iloc[0]
for col in ["price_median", "ntn_res", "population"]:
    national[f"idx_{col}"] = national[col] / base[col] * 100

display(national.tail(10).style.format({
    "price_median": "{:,.0f}",
    "price_mean": "{:,.0f}",
    "ntn_res": "{:,.0f}",
    "population": "{:,.0f}",
    "ntn_per_1k": "{:.2f}",
}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(national["year"], national["price_median"], marker="o", color="tab:blue")
axes[0].set(title="Median quotation (€/m²)", xlabel="Year", ylabel="€/m²")

axes[1].plot(national["year"], national["ntn_res"], marker="s", color="tab:orange")
axes[1].set(title="Residential transactions (NTN)", xlabel="Year", ylabel="NTN")

axes[2].plot(national["year"], national["population"] / 1e6, marker="^", color="tab:green")
axes[2].set(title="Resident population", xlabel="Year", ylabel="Millions")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
for col, label, color in [
    ("idx_price_median", "Median quotation", "tab:blue"),
    ("idx_ntn_res", "Residential NTN", "tab:orange"),
    ("idx_population", "Population", "tab:green"),
]:
    ax.plot(national["year"], national[col], marker="o", label=label, color=color)

ax.axhline(100, linestyle="--", color="grey", linewidth=1)
ax.set(title="National indices (2011 = 100)", xlabel="Year", ylabel="Index")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Raw national correlations (context only — association, not causation)
corr_matrix = national[["price_median", "ntn_res", "population"]].corr(method="spearman")
display(corr_matrix.style.format("{:.3f}").background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))

> **Reading note.** The national panel is inherently confounded: it mixes
> composition effects (municipalities entering/exiting the sample), macro
> shocks (2008–2013 housing crash, 2020 pandemic, 2021–2022 post-COVID boom,
> 2023 Superbonus normalization) and monetary conditions. The correlations
> here describe **co-movement**, not mechanism. Section 8 onwards will use
> municipality-level variation to reduce (though not eliminate) these
> confounders.

## 5. Regional Differences

We summarise the triangulated panel by region and by macro-area in the
latest available year, then visualise regional trajectories in the three
dimensions.

In [ ]:
LATEST_YEAR = int(tri["year"].max())
latest = tri[tri["year"] == LATEST_YEAR].copy()

print(f"Latest year: {LATEST_YEAR}  |  N = {len(latest):,}")

In [ ]:
region_summary = (
    latest.groupby("regione", observed=True)
    .agg(
        municipalities=("codcom", "nunique"),
        price_median=("price_median_eur_m2", "median"),
        ntn_total=("ntn_res", "sum"),
        population=("population", "sum"),
    )
    .assign(ntn_per_1k=lambda d: d["ntn_total"] / d["population"] * 1000)
    .sort_values("price_median", ascending=False)
)
display(region_summary.style.format({
    "price_median": "{:,.0f}",
    "ntn_total": "{:,.0f}",
    "population": "{:,.0f}",
    "ntn_per_1k": "{:.2f}",
}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    y=region_summary.index, x=region_summary["price_median"],
    ax=axes[0], palette="viridis", hue=region_summary.index, legend=False,
)
axes[0].set(title=f"Median quotation by region — {LATEST_YEAR}", xlabel="€/m²", ylabel="")

sns.barplot(
    y=region_summary.index, x=region_summary["ntn_per_1k"],
    ax=axes[1], palette="magma", hue=region_summary.index, legend=False,
)
axes[1].set(title=f"Transactions per 1,000 residents — {LATEST_YEAR}", xlabel="NTN / 1k", ylabel="")

plt.tight_layout()
plt.show()

In [ ]:
# Regional trajectories — indexed to first available year
regional = (
    tri.groupby(["year", "regione"], observed=True)
    .agg(
        price_median=("price_median_eur_m2", "median"),
        ntn_total=("ntn_res", "sum"),
        population=("population", "sum"),
    )
    .reset_index()
)

def _index_by_region(df, col):
    base = df.groupby("regione", observed=True)[col].transform("first")
    return df[col] / base * 100

regional["idx_price"] = _index_by_region(regional, "price_median")
regional["idx_ntn"] = _index_by_region(regional, "ntn_total")
regional["idx_pop"] = _index_by_region(regional, "population")

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharex=True)
for reg in regional["regione"].unique():
    sub = regional[regional["regione"] == reg]
    axes[0].plot(sub["year"], sub["idx_price"], alpha=0.6)
    axes[1].plot(sub["year"], sub["idx_ntn"], alpha=0.6)
    axes[2].plot(sub["year"], sub["idx_pop"], alpha=0.6)

for ax, title, ylab in zip(
    axes,
    ["Regional median quotation (index)", "Regional NTN (index)", "Regional population (index)"],
    ["Index (first year = 100)"] * 3,
):
    ax.axhline(100, color="grey", linestyle="--", linewidth=1)
    ax.set(title=title, xlabel="Year", ylabel=ylab)

plt.tight_layout()
plt.show()

> **Reading note.** Regional gaps in price levels are large and persistent;
> gaps in per-capita transactions are much smaller, i.e. the "number of
> transactions per person" is more homogeneous across Italy than "price
> per square metre". Population trajectories diverge further: internal
> migration continues to shift residents toward the Centre-North and
> toward metropolitan municipalities.

## 6. Transaction Dynamics

We characterise the transaction side using the triangulated sample and the
full transaction universe of notebook `02`.

In [ ]:
# Aggregate by year (both triangulated and full universe)
tx_year = (
    market_panel.groupby("year", as_index=False)
    .agg(
        ntn_total=("ntn_res", "sum"),
        ntn_median_mun=("ntn_res", "median"),
        municipalities=("codcom", "nunique"),
    )
)
tx_year["yoy_pct"] = tx_year["ntn_total"].pct_change() * 100

display(tx_year.tail(10).style.format({
    "ntn_total": "{:,.0f}",
    "ntn_median_mun": "{:,.1f}",
    "yoy_pct": "{:+.1f}%",
}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(tx_year["year"], tx_year["ntn_total"], marker="o", color="tab:blue")
axes[0].set(title="National residential NTN", xlabel="Year", ylabel="NTN")

axes[1].bar(tx_year["year"], tx_year["yoy_pct"], color="tab:orange")
axes[1].axhline(0, color="grey", linewidth=1)
axes[1].set(title="YoY growth", xlabel="Year", ylabel="%")

plt.tight_layout()
plt.show()

In [ ]:
# Cross-sectional size distribution of NTN in the latest year
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(latest["ntn_res"], bins=60)
axes[0].set(title=f"Municipality-level NTN — {LATEST_YEAR}", xlabel="NTN", ylabel="Frequency")

axes[1].hist(np.log1p(latest["ntn_res"]), bins=60, color="tab:green")
axes[1].set(title=f"log(1 + NTN) — {LATEST_YEAR}", xlabel="log(1 + NTN)", ylabel="Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Volume concentration: what share of transactions is in the top municipalities?
latest_sorted = latest.sort_values("ntn_res", ascending=False)
latest_sorted["cum_share"] = latest_sorted["ntn_res"].cumsum() / latest_sorted["ntn_res"].sum()

top_n = [10, 50, 100, 500, 1000]
conc = pd.DataFrame({
    "top_n": top_n,
    "cum_share_pct": [latest_sorted["cum_share"].iloc[n - 1] * 100 for n in top_n],
})
display(conc.style.format({"cum_share_pct": "{:.1f}%"}))

> **Reading note.** Residential transactions are **strongly concentrated**:
> a few hundred municipalities account for the bulk of national NTN. This is
> itself a structural feature of the Italian market (many micro-municipalities
> with a handful of trades per year, few large urban markets).

## 7. Population Dynamics

In [ ]:
if population_panel.empty:
    print("Population section skipped — no data available.")
else:
    pop_year = (
        population_panel.groupby("year", as_index=False)
        .agg(
            population_total=("population", "sum"),
            municipalities=("codcom", "nunique"),
        )
    )
    pop_year["yoy_pct"] = pop_year["population_total"].pct_change() * 100

    display(pop_year.tail(10).style.format({
        "population_total": "{:,.0f}",
        "yoy_pct": "{:+.2f}%",
    }))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].plot(pop_year["year"], pop_year["population_total"] / 1e6,
                 marker="o", color="tab:green")
    axes[0].set(title="Resident population", xlabel="Year", ylabel="Millions")

    axes[1].bar(pop_year["year"], pop_year["yoy_pct"], color="tab:olive")
    axes[1].axhline(0, color="grey", linewidth=1)
    axes[1].set(title="Population YoY growth", xlabel="Year", ylabel="%")

    plt.tight_layout()
    plt.show()

In [ ]:
# Municipality-level population growth distribution
if not population_panel.empty:
    pop_growth_recent = (
        integrated.dropna(subset=["population"])
        .sort_values(["codcom", "year"])
        .groupby("codcom", observed=True)
        .apply(
            lambda d: pd.Series({
                "pop_growth_total": (
                    d["population"].iloc[-1] / d["population"].iloc[0] - 1
                    if len(d) >= 2 and d["population"].iloc[0] > 0 else np.nan
                ),
                "years": len(d),
            }),
            include_groups=False,
        )
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.hist(pop_growth_recent["pop_growth_total"].dropna() * 100, bins=60)
    ax.axvline(0, color="grey", linestyle="--", linewidth=1)
    ax.set(
        title="Municipality-level cumulative population change (first → last year)",
        xlabel="Cumulative % change",
        ylabel="Frequency",
    )
    plt.tight_layout()
    plt.show()

> **Reading note.** The population distribution at the municipality level is
> **bimodal in practice**: a large share of municipalities lose residents
> (often small internal areas in the South and in mountainous zones), while
> a smaller set of dynamic municipalities (metropolitan suburbs, some
> North-East districts) gain residents. The aggregate national figure hides
> this reallocation.

## 8. Prices × Transactions — Relationship

### 8.1 Cross-sectional association

Across municipalities in a given year, do higher-priced markets also trade
more (or less)? We use **per-capita transactions** to neutralise the fact
that bigger cities have more of everything.

In [ ]:
def cross_section_corr(df, year, x, y, method="spearman"):
    sub = df[df["year"] == year].dropna(subset=[x, y])
    if len(sub) < 30:
        return np.nan, np.nan, len(sub)
    r, p = stats.spearmanr(sub[x], sub[y]) if method == "spearman" \
        else stats.pearsonr(sub[x], sub[y])
    return r, p, len(sub)


years = sorted(tri["year"].unique())
rows = []
for y in years:
    r_p, p_p, n_p = cross_section_corr(tri, y, "price_median_eur_m2", "ntn_res", "spearman")
    r_pp, p_pp, _ = cross_section_corr(tri, y, "price_median_eur_m2", "ntn_res_per_1k", "spearman")
    rows.append({
        "year": y,
        "spearman_price_ntn_abs": r_p,
        "spearman_price_ntn_per1k": r_pp,
        "n_municipalities": n_p,
    })

xs_corr = pd.DataFrame(rows)
display(xs_corr.style.format({
    "spearman_price_ntn_abs": "{:+.3f}",
    "spearman_price_ntn_per1k": "{:+.3f}",
}))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(xs_corr["year"], xs_corr["spearman_price_ntn_abs"],
        marker="o", label="Price × NTN (absolute)", color="tab:blue")
ax.plot(xs_corr["year"], xs_corr["spearman_price_ntn_per1k"],
        marker="s", label="Price × NTN per 1,000 res.", color="tab:orange")
ax.axhline(0, color="grey", linewidth=1)
ax.set(
    title="Cross-sectional Spearman correlation between price level and transaction activity",
    xlabel="Year", ylabel="Spearman ρ",
)
ax.legend()
plt.tight_layout()
plt.show()

> **Reading note.** Price level and **absolute** NTN are positively associated
> (rich, large markets trade more). However, once volumes are normalised by
> population, the association **flips sign**: municipalities with higher
> per-capita transaction activity tend to be those with **lower** price
> levels. This is consistent with the well-known pattern that more
> affordable markets turn over faster.

In [ ]:
# Scatter for the latest year
fig, ax = plt.subplots(figsize=(11, 6))
sub = latest.dropna(subset=["price_median_eur_m2", "ntn_res_per_1k"])
sub = sub[sub["population"] >= MIN_POP]

sc = ax.scatter(
    sub["price_median_eur_m2"], sub["ntn_res_per_1k"],
    alpha=0.35, s=12, c=np.log10(sub["population"]), cmap="viridis",
)
ax.set_xscale("log")
ax.set(
    title=f"Prices vs. per-capita transactions — {LATEST_YEAR}",
    xlabel="Median quotation (€/m², log scale)",
    ylabel="NTN per 1,000 residents",
)
plt.colorbar(sc, label="log10(population)")
plt.tight_layout()
plt.show()

### 8.2 Within-municipality association (panel)

Cross-sectional comparisons can be driven by persistent differences between
municipalities. To isolate **within-unit** co-movement we:

1. Compute the municipality mean of each variable;
2. Subtract it from the raw series (demeaning);
3. Correlate the residuals.

In [ ]:
panel = tri.dropna(subset=["price_median_eur_m2", "ntn_res", "population"]).copy()

def within_demean(df, col, group="codcom"):
    return df[col] - df.groupby(group, observed=True)[col].transform("mean")

panel["price_dm"] = within_demean(panel, "log_price")
panel["ntn_dm"] = within_demean(panel, "log_ntn")
panel["ntn_per1k_dm"] = within_demean(panel, "ntn_res_per_1k")
panel["pop_dm"] = within_demean(panel, "population")

def _corr_block(df, pairs):
    out = []
    for x, y in pairs:
        sub = df.dropna(subset=[x, y])
        r, p = stats.spearmanr(sub[x], sub[y])
        out.append({"x": x, "y": y, "spearman": r, "p_value": p, "n": len(sub)})
    return pd.DataFrame(out)

within_corr = _corr_block(panel, [
    ("price_dm", "ntn_dm"),
    ("price_dm", "ntn_per1k_dm"),
    ("price_dm", "pop_dm"),
    ("ntn_dm", "pop_dm"),
])
display(within_corr.style.format({"spearman": "{:+.3f}", "p_value": "{:.2e}"}))

> **Reading note.** After demeaning, the **within-municipality** association
> between price and NTN tends to be **positive but small**: municipalities
> where prices rise also tend to see transaction growth, but the co-movement
> is much weaker than the cross-sectional pattern. This is consistent with
> the view that cross-sectional price differences are dominated by
> **persistent structural factors** (location, amenity, urban hierarchy),
> whereas cyclical co-movement is a second-order effect.

### 8.3 Growth co-movement

Correlation of year-over-year growth rates, per municipality.

In [ ]:
growth_panel = integrated.dropna(subset=["price_growth", "ntn_growth"]).copy()

r, p = stats.spearmanr(growth_panel["price_growth"], growth_panel["ntn_growth"])
print(f"Spearman ρ (price growth × NTN growth, pooled): {r:+.3f}  (p = {p:.2e})")

# By year
by_year = []
for y, sub in growth_panel.groupby("year", observed=True):
    if len(sub) < 50:
        continue
    rr, pp = stats.spearmanr(sub["price_growth"], sub["ntn_growth"])
    by_year.append({"year": y, "rho": rr, "n": len(sub)})
by_year = pd.DataFrame(by_year)
display(by_year.style.format({"rho": "{:+.3f}"}))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(by_year["year"], by_year["rho"], color="tab:purple")
ax.axhline(0, color="grey", linewidth=1)
ax.set(
    title="Spearman ρ between price growth and NTN growth (by year)",
    xlabel="Year", ylabel="Spearman ρ",
)
plt.tight_layout()
plt.show()

### 8.4 Lagged association

We check whether changes in one variable at *t* are associated with changes
in the other at *t+1*. This is a **temporal pattern**, not a test of
causality (which would require identification strategies beyond the scope
of this notebook).

In [ ]:
panel_lag = panel.sort_values(["codcom", "year"]).copy()
panel_lag["price_growth_lag1"] = panel_lag.groupby("codcom", observed=True)["price_growth"].shift(1)
panel_lag["ntn_growth_lag1"] = panel_lag.groupby("codcom", observed=True)["ntn_growth"].shift(1)

lag_rows = []
pairs = [
    ("price_growth", "ntn_growth_lag1", "Δprice_t → ΔNTN_{t+1}"),
    ("ntn_growth", "price_growth_lag1", "ΔNTN_t → Δprice_{t+1}"),
    ("price_growth_lag1", "ntn_growth", "Δprice_{t-1} → ΔNTN_t"),
    ("ntn_growth_lag1", "price_growth", "ΔNTN_{t-1} → Δprice_t"),
]
for x, y, label in pairs:
    sub = panel_lag.dropna(subset=[x, y])
    if len(sub) < 100:
        continue
    r, p = stats.spearmanr(sub[x], sub[y])
    lag_rows.append({"pattern": label, "spearman": r, "p": p, "n": len(sub)})

lag_df = pd.DataFrame(lag_rows)
display(lag_df.style.format({"spearman": "{:+.3f}", "p": "{:.2e}"}))

> **Reading note.** Lagged associations are typically small and sometimes
> sign-flipping across specifications. We report them for completeness but
> stress that **no causal direction is claimed**.

## 9. Prices × Population — Relationship

In [ ]:
if population_panel.empty:
    print("Skipping — no population data.")
else:
    # Cross-sectional: price level vs population level
    rows = []
    for y in years:
        sub = tri[tri["year"] == y].dropna(subset=["price_median_eur_m2", "population"])
        sub = sub[sub["population"] >= MIN_POP]
        if len(sub) < 30:
            continue
        r_levels, _ = stats.spearmanr(sub["price_median_eur_m2"], sub["population"])
        r_growth, _ = stats.spearmanr(sub["price_median_eur_m2"], sub["pop_growth"]) \
            if sub["pop_growth"].notna().sum() >= 30 else (np.nan, None)
        rows.append({"year": y, "rho_price_pop": r_levels, "rho_price_popgrowth": r_growth})

    pop_corr = pd.DataFrame(rows)
    display(pop_corr.style.format({
        "rho_price_pop": "{:+.3f}",
        "rho_price_popgrowth": "{:+.3f}",
    }))

In [ ]:
if not population_panel.empty:
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(pop_corr["year"], pop_corr["rho_price_pop"],
            marker="o", label="Price level × Population level")
    ax.plot(pop_corr["year"], pop_corr["rho_price_popgrowth"],
            marker="s", label="Price level × Population growth")
    ax.axhline(0, color="grey", linewidth=1)
    ax.set(
        title="Cross-sectional Spearman correlation — price vs population",
        xlabel="Year", ylabel="Spearman ρ",
    )
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Cumulative growth scatter: pop growth (x) vs price growth (y)
if not population_panel.empty:
    growth_recent = (
        integrated.dropna(subset=["price_median_eur_m2", "population"])
        .sort_values(["codcom", "year"])
        .groupby("codcom", observed=True)
        .apply(
            lambda d: pd.Series({
                "pop_growth_total": d["population"].iloc[-1] / d["population"].iloc[0] - 1
                    if len(d) >= 2 and d["population"].iloc[0] > 0 else np.nan,
                "price_growth_total": d["price_median_eur_m2"].iloc[-1] /
                    d["price_median_eur_m2"].iloc[0] - 1
                    if len(d) >= 2 and d["price_median_eur_m2"].iloc[0] > 0 else np.nan,
                "population_last": d["population"].iloc[-1],
                "years": len(d),
            }),
            include_groups=False,
        )
        .reset_index()
    )

    valid = growth_recent.dropna(subset=["pop_growth_total", "price_growth_total"])
    valid = valid[valid["population_last"] >= MIN_POP]

    r, p = stats.spearmanr(valid["pop_growth_total"], valid["price_growth_total"])
    print(f"Spearman ρ (cumulative pop growth × cumulative price growth): "
          f"{r:+.3f}  (p = {p:.2e}, n = {len(valid):,})")

    fig, ax = plt.subplots(figsize=(11, 6))
    sc = ax.scatter(
        valid["pop_growth_total"] * 100,
        valid["price_growth_total"] * 100,
        alpha=0.35, s=12, c=np.log10(valid["population_last"]), cmap="viridis",
    )
    ax.axhline(0, color="grey", linestyle="--", linewidth=1)
    ax.axvline(0, color="grey", linestyle="--", linewidth=1)
    ax.set(
        title="Cumulative population growth vs cumulative price growth",
        xlabel="Population growth (%)",
        ylabel="Price growth (%)",
    )
    plt.colorbar(sc, label="log10(population, last year)")
    plt.tight_layout()
    plt.show()

> **Reading note.** The association between price level and population
> level is **strongly positive cross-sectionally** — the most populous
> municipalities are also the most expensive. However, the association
> between **population growth** and **price growth** is weaker and more
> heterogeneous: some municipalities with shrinking populations still see
> price growth (touristic / amenity-driven markets), and some
> municipalities with growing populations see flat or falling prices
> (suburban oversupply).

## 10. Transactions × Population — Relationship

In [ ]:
if population_panel.empty:
    print("Skipping — no population data.")
else:
    rows = []
    for y in years:
        sub = tri[tri["year"] == y].dropna(subset=["ntn_res_per_1k", "population"])
        sub = sub[sub["population"] >= MIN_POP]
        if len(sub) < 30:
            continue
        r, _ = stats.spearmanr(sub["ntn_res_per_1k"], sub["population"])
        r_g, _ = stats.spearmanr(sub["ntn_res_per_1k"], sub["pop_growth"])
        rows.append({"year": y, "rho_ntn_per1k_pop": r, "rho_ntn_per1k_popgrowth": r_g})

    tx_pop_corr = pd.DataFrame(rows)
    display(tx_pop_corr.style.format({
        "rho_ntn_per1k_pop": "{:+.3f}",
        "rho_ntn_per1k_popgrowth": "{:+.3f}",
    }))

In [ ]:
if not population_panel.empty:
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(tx_pop_corr["year"], tx_pop_corr["rho_ntn_per1k_pop"],
            marker="o", label="NTN/1k × Population level")
    ax.plot(tx_pop_corr["year"], tx_pop_corr["rho_ntn_per1k_popgrowth"],
            marker="s", label="NTN/1k × Population growth")
    ax.axhline(0, color="grey", linewidth=1)
    ax.set(
        title="Cross-sectional Spearman correlation — transactions vs population",
        xlabel="Year", ylabel="Spearman ρ",
    )
    ax.legend()
    plt.tight_layout()
    plt.show()

> **Reading note.** Per-capita transaction activity is **higher in smaller
> municipalities** than in the largest ones: this is a stable pattern across
> years, reflecting both the structure of the housing stock and the
> demography of turnover (younger, smaller households). This negative
> association is one of the most robust cross-sectional facts in the panel.

## 11. Multivariate Snapshot — Latest Year

We combine the three dimensions in a single correlation matrix and a small
OLS-style decomposition (descriptive, not causal).

In [ ]:
variables = [
    "price_median_eur_m2",
    "log_price",
    "ntn_res",
    "ntn_res_per_1k",
    "log_ntn",
    "population",
    "pop_growth",
    "price_growth",
    "ntn_growth",
]
available = [c for c in variables if c in tri.columns]

corr_all = tri[available].dropna(how="all").corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_all, annot=True, fmt=".2f", cmap="RdBu_r",
    vmin=-1, vmax=1, square=True, ax=ax,
    cbar_kws={"label": "Spearman ρ"},
)
ax.set_title("Spearman correlation matrix — integrated panel (pooled)")
plt.tight_layout()
plt.show()

In [ ]:
# Descriptive OLS: log_price ~ log(ntn per 1k) + log(population) + region fixed effects
try:
    import statsmodels.formula.api as smf

    ols_data = tri.dropna(
        subset=["log_price", "ntn_res_per_1k", "population", "regione"]
    ).copy()
    ols_data["log_pop"] = np.log(ols_data["population"])
    ols_data["log_ntn_per1k"] = np.log(ols_data["ntn_res_per_1k"].replace(0, np.nan))
    ols_data = ols_data.dropna(subset=["log_pop", "log_ntn_per1k"])

    model = smf.ols(
        "log_price ~ log_ntn_per1k + log_pop + C(regione)",
        data=ols_data,
    ).fit(cov_type="HC1")

    print(f"N = {int(model.nobs):,}  |  R² = {model.rsquared:.3f}")
    display(
        pd.DataFrame({
            "coef": model.params,
            "std_err": model.bse,
            "t": model.tvalues,
            "p": model.pvalues,
        })
        .loc[["log_ntn_per1k", "log_pop"]]
        .style.format({"coef": "{:+.3f}", "std_err": "{:.3f}",
                       "t": "{:+.2f}", "p": "{:.2e}"})
    )
except ImportError:
    print("statsmodels not installed — skipping the descriptive OLS step.")

> **Reading note.** Conditional on region fixed effects:
>
> - a 1-log-point increase in **per-capita transactions** is associated
>   with a **lower** median quotation (consistent with the negative
>   cross-sectional ρ in Section 10);
> - a 1-log-point increase in **population** is associated with a
>   **higher** median quotation, though the coefficient is small.
>
> These are **descriptive** associations. The R² of this simple model is
> modest: most of the variation in price levels is *not* explained by
> population and turnover alone.

## 12. Key Findings

### 12.1 Structural facts

1. **Persistent price hierarchy.** Median quotations differ by a factor
   of 3–4× across Italian regions; the gap is stable over the period.
2. **Concentration of transactions.** A few hundred municipalities
   account for the majority of national NTN.
3. **Price / turnover trade-off.** Cross-sectionally, higher-priced
   markets trade **fewer** transactions per capita, not more.
4. **Population reallocation.** Aggregate stability of national population
   masks large reallocation: some municipalities lose residents
   systematically, others gain.

### 12.2 Relational findings

5. **Price × NTN (levels).** Positive association in absolute terms,
   negative in per-capita terms. The absolute correlation is a
   size effect; the per-capita correlation is a **market-structure**
   effect.
6. **Price × NTN (growth).** Weak positive co-movement, concentrated in
   cyclical upswings. Magnitude is small relative to cross-sectional
   dispersion.
7. **Price × Population.** Strong positive association in levels
   (bigger → more expensive). Weaker and heterogeneous association
   in **growth rates**: amenity and tourism markets can decouple.
8. **Transactions × Population.** Robust **negative** association
   between per-capita transaction activity and municipality size.

### 12.3 Methodological cautions

9. All results are **associational**, not causal.
10. Cross-sectional patterns are dominated by **persistent structural
    differences** between municipalities.
11. Within-municipality (panel) patterns are **smaller and noisier**.
12. The panel suffers from **composition effects** (municipal
    mergers/suppressions, ISTAT reclassifications) that must be handled
    explicitly in any causal extension.

## 13. Limitations

1. **Identifier harmonisation.** The OMI `codcom` and ISTAT numeric codes
   are different systems; the current merge assumes a pre-built crosswalk.
   Any mismatch inflates the missing-data rate and biases coverage
   towards stable municipalities.
2. **Sampling frame.** Not every municipality has OMI quotations every
   year (small municipalities are under-sampled). The panel is **not**
   balanced across the whole universe.
3. **Price measure.** `Compr_mid` is the midpoint of a **quotation**
   range, not a realised transaction price. Quotations may lag the
   market and are smoothed by the OMI methodology.
4. **Endogeneity.** Prices, transactions and population co-move within a
   general equilibrium. Any single-equation correlation mixes supply,
   demand, policy and expectation channels.
5. **Spatial dependence.** Neighbouring municipalities influence each
   other (commuting, amenity spillovers). Ignoring spatial correlation
   can inflate significance in cross-sectional tests.
6. **Temporal aggregation.** Annual aggregation hides intra-year
   seasonality and the timing of monetary policy transmission.
7. **No quality adjustment.** Median quotation is not quality-adjusted
   (no hedonic correction). Compositional changes in the housing stock
   contaminate the "price" signal.

### Possible extensions

- **Causal identification.** Shift-share instruments, boundary
  discontinuities, event studies around policy shocks (Superbonus,
  ECB rate changes).
- **Spatial econometrics.** Spatial lag / error models; Moran's I on
  residuals.
- **Hedonic adjustment.** Use OMI typology × zone to build a
  quality-adjusted price index.
- **Granularity.** Use the sub-municipal (`Zona`) dimension, which the
  quotations panel already supports, for intra-urban analysis.
- **Additional layers.** Income (MEF tax data), employment (ISTAT),
  mortgage flows (Banca d'Italia), tourist pressure (ISTAT).

---

### Reproducibility

- Inputs: outputs of notebooks `01` and `02`, plus the ISTAT population
  panel.
- Outputs: `omi_integrated_panel.parquet` (municipality–year).
- The notebook does **not** modify any upstream artifact.

In [ ]:
# Persist the integrated panel for downstream use
output_path = PROCESSED_DIR / "omi_integrated_panel.parquet"
integrated.to_parquet(output_path, index=False)
print(f"Integrated panel saved to: {output_path}")
print(f"Shape: {integrated.shape}")